# AAPL TCN Backtesting

## Objective

Evaluate the trading performance of the trained AAPL TCN
using the `backtesting.py` framework.

The strategy follows the research-paper investment logic:

- Predicted Rise → Buy / Hold
- Predicted Fall → Exit position

The trained TCN model is loaded from the saved checkpoint.
The test period is used for the out-of-sample trading evaluation.

No model retraining is performed in this notebook.

In [27]:
import os
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from backtesting import Backtest, Strategy

warnings.filterwarnings("ignore")

print("Imports successful.")

Imports successful.


In [28]:
PROJECT_ROOT = os.path.abspath("..")

DATA_DIR = os.path.join(
    PROJECT_ROOT,
    "data"
)

RAW_DATA_DIR = os.path.join(
    DATA_DIR,
    "raw"
)

PROCESSED_DATA_DIR = os.path.join(
    DATA_DIR,
    "processed"
)

MODEL_DIR = os.path.join(
    PROJECT_ROOT,
    "models"
)

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "aapl_tcn.pt"
)

print("=" * 60)
print("PROJECT PATHS")
print("=" * 60)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("Processed data:", PROCESSED_DATA_DIR)
print("Model:", MODEL_PATH)

PROJECT PATHS
Project root: /Users/senuka/Documents/Projects/My Projects/quantitative-research-stock-trend
Raw data: /Users/senuka/Documents/Projects/My Projects/quantitative-research-stock-trend/data/raw
Processed data: /Users/senuka/Documents/Projects/My Projects/quantitative-research-stock-trend/data/processed
Model: /Users/senuka/Documents/Projects/My Projects/quantitative-research-stock-trend/models/aapl_tcn.pt


In [29]:
assert os.path.exists(
    MODEL_PATH
), f"Missing trained model: {MODEL_PATH}"

print("=" * 60)
print("REQUIRED ARTIFACT CHECK")
print("=" * 60)

print("TCN model exists: PASSED")

REQUIRED ARTIFACT CHECK
TCN model exists: PASSED


In [30]:
AAPL_DATA_PATH = os.path.join(
    RAW_DATA_DIR,
    "AAPL_1y_daily.csv"
)

assert os.path.exists(
    AAPL_DATA_PATH
), f"Missing AAPL data: {AAPL_DATA_PATH}"

aapl = pd.read_csv(
    AAPL_DATA_PATH
)

print("=" * 60)
print("AAPL DATA LOADED")
print("=" * 60)

print("Shape:", aapl.shape)
print("Columns:", list(aapl.columns))

AAPL DATA LOADED
Shape: (251, 6)
Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']


In [31]:
aapl["Date"] = pd.to_datetime(
    aapl["Date"]
)

aapl = aapl.sort_values(
    "Date"
).reset_index(drop=True)

REQUIRED_COLUMNS = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Volume"
]

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in aapl.columns
]

print("Missing columns:", missing_columns)

assert len(missing_columns) == 0

assert aapl["Date"].is_monotonic_increasing

assert aapl[
    REQUIRED_COLUMNS[1:]
].notna().all().all()

print("=" * 60)
print("AAPL DATA VALIDATION")
print("=" * 60)

print("OHLCV columns: PASSED")
print("Chronological order: PASSED")
print("Missing OHLCV values: PASSED")

print(
    "Date range:",
    aapl["Date"].min().date(),
    "→",
    aapl["Date"].max().date()
)

Missing columns: []
AAPL DATA VALIDATION
OHLCV columns: PASSED
Chronological order: PASSED
Missing OHLCV values: PASSED
Date range: 2025-08-18 → 2026-08-17


In [32]:
# DEFINE MODEL FEATURES

FEATURE_COLUMNS = [
    "Open",
    "High",
    "Low",
    "Close",
    "ATR",
    "EMA20",
    "MOM6",
    "CCI",
    "MACD"
]

print("=" * 60)
print("MODEL FEATURES")
print("=" * 60)

for index, feature in enumerate(
    FEATURE_COLUMNS,
    start=1
):
    print(f"{index}. {feature}")

print()
print("Feature count:", len(FEATURE_COLUMNS))

assert len(FEATURE_COLUMNS) == 9

print("Feature count validation: PASSED")

MODEL FEATURES
1. Open
2. High
3. Low
4. Close
5. ATR
6. EMA20
7. MOM6
8. CCI
9. MACD

Feature count: 9
Feature count validation: PASSED


In [33]:
# CALCULATE MODEL INDICATORS

# EMA20
aapl["EMA20"] = (
    aapl["Close"]
    .ewm(
        span=20,
        adjust=False
    )
    .mean()
)


# MOM6
aapl["MOM6"] = (
    aapl["Close"]
    - aapl["Close"].shift(6)
)


# True Range
previous_close = (
    aapl["Close"].shift(1)
)

true_range = pd.concat(
    [
        aapl["High"] - aapl["Low"],

        (
            aapl["High"]
            - previous_close
        ).abs(),

        (
            aapl["Low"]
            - previous_close
        ).abs()
    ],
    axis=1
).max(axis=1)


# ATR
aapl["ATR"] = (
    true_range
    .rolling(window=14)
    .mean()
)


# CCI
typical_price = (
    aapl["High"]
    + aapl["Low"]
    + aapl["Close"]
) / 3

cci_period = 20

tp_mean = (
    typical_price
    .rolling(cci_period)
    .mean()
)

mean_deviation = (
    typical_price
    .rolling(cci_period)
    .apply(
        lambda x: np.mean(
            np.abs(
                x - np.mean(x)
            )
        ),
        raw=True
    )
)

aapl["CCI"] = (
    (typical_price - tp_mean)
    / (0.015 * mean_deviation)
)


# MACD
ema12 = (
    aapl["Close"]
    .ewm(
        span=12,
        adjust=False
    )
    .mean()
)

ema26 = (
    aapl["Close"]
    .ewm(
        span=26,
        adjust=False
    )
    .mean()
)

aapl["MACD"] = (
    ema12 - ema26
)


print("=" * 60)
print("INDICATORS CALCULATED")
print("=" * 60)

print(
    aapl[
        [
            "ATR",
            "EMA20",
            "MOM6",
            "CCI",
            "MACD"
        ]
    ].isna().sum()
)

INDICATORS CALCULATED
ATR      13
EMA20     0
MOM6      6
CCI      19
MACD      0
dtype: int64


In [34]:
# REMOVE INDICATOR WARM-UP ROWS

INDICATOR_COLUMNS = [
    "ATR",
    "EMA20",
    "MOM6",
    "CCI",
    "MACD"
]

aapl_model = aapl.dropna(
    subset=INDICATOR_COLUMNS
).reset_index(drop=True)

print("=" * 60)
print("INDICATOR WARM-UP CHECK")
print("=" * 60)

print("Rows before warm-up:", len(aapl))
print("Rows after warm-up:", len(aapl_model))

print("\nRemaining missing indicator values:")

print(
    aapl_model[
        INDICATOR_COLUMNS
    ].isna().sum()
)

assert (
    aapl_model[
        INDICATOR_COLUMNS
    ].isna().sum().sum()
    == 0
)

print("\nIndicator warm-up validation: PASSED")

INDICATOR WARM-UP CHECK
Rows before warm-up: 251
Rows after warm-up: 232

Remaining missing indicator values:
ATR      0
EMA20    0
MOM6     0
CCI      0
MACD     0
dtype: int64

Indicator warm-up validation: PASSED


In [35]:
# FINAL MODEL FEATURE VALIDATION

FEATURE_COLUMNS = [
    "Open",
    "High",
    "Low",
    "Close",
    "ATR",
    "EMA20",
    "MOM6",
    "CCI",
    "MACD"
]

missing_features = [
    feature
    for feature in FEATURE_COLUMNS
    if feature not in aapl_model.columns
]

print("=" * 60)
print("MODEL FEATURE VALIDATION")
print("=" * 60)

print("Missing features:", missing_features)

assert len(missing_features) == 0

missing_values = (
    aapl_model[
        FEATURE_COLUMNS
    ].isna().sum()
)

print("\nMissing values:")
print(missing_values)

assert missing_values.sum() == 0

print("\nFeature count:", len(FEATURE_COLUMNS))

assert len(FEATURE_COLUMNS) == 9

print("Feature validation: PASSED")

MODEL FEATURE VALIDATION
Missing features: []

Missing values:
Open     0
High     0
Low      0
Close    0
ATR      0
EMA20    0
MOM6     0
CCI      0
MACD     0
dtype: int64

Feature count: 9
Feature validation: PASSED


In [36]:
# TCN ARCHITECTURE

class AAPLTCN(nn.Module):

    def __init__(
        self,
        num_features=9,
        num_classes=2
    ):
        super().__init__()

        self.tcn = nn.Sequential(

            TemporalBlock(
                in_channels=num_features,
                out_channels=32,
                kernel_size=3,
                dilation=1,
                dropout=0.2
            ),

            TemporalBlock(
                in_channels=32,
                out_channels=32,
                kernel_size=3,
                dilation=2,
                dropout=0.2
            )
        )

        self.classifier = nn.Linear(
            32,
            num_classes
        )

    def forward(self, x):

        x = x.transpose(1, 2)

        x = self.tcn(x)

        x = x[:, :, -1]

        return self.classifier(x)


print("AAPLTCN architecture defined.")

AAPLTCN architecture defined.


In [38]:
class TemporalBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
        dilation=1,
        dropout=0.2
    ):
        super().__init__()

        padding = (
            kernel_size - 1
        ) * dilation

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(
            dropout
        )

        self.residual = (
            nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=1
            )
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):

        residual = self.residual(x)

        out = self.conv1(x)

        out = out[:, :, :x.size(2)]

        out = self.relu(out)

        out = self.dropout(out)

        out = self.conv2(out)

        out = out[:, :, :x.size(2)]

        out = self.relu(out)

        out = self.dropout(out)

        return self.relu(
            out + residual
        )


class AAPLTCN(nn.Module):

    def __init__(
        self,
        num_features=9,
        num_classes=2
    ):
        super().__init__()

        self.tcn = nn.Sequential(

            TemporalBlock(
                in_channels=num_features,
                out_channels=32,
                kernel_size=3,
                dilation=1,
                dropout=0.2
            ),

            TemporalBlock(
                in_channels=32,
                out_channels=32,
                kernel_size=3,
                dilation=2,
                dropout=0.2
            )
        )

        self.classifier = nn.Linear(
            32,
            num_classes
        )

    def forward(self, x):

        x = x.transpose(
            1,
            2
        )

        x = self.tcn(x)

        x = x[:, :, -1]

        return self.classifier(x)


print("TCN architecture defined.")

TCN architecture defined.


In [39]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = AAPLTCN(
    num_features=9,
    num_classes=2
).to(device)

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device,
        weights_only=True
    )
)

model.eval()

print("=" * 60)
print("FROZEN TCN MODEL")
print("=" * 60)

print("Model path:", MODEL_PATH)
print("Device:", device)
print("Model loaded: PASSED")

FROZEN TCN MODEL
Model path: /Users/senuka/Documents/Projects/My Projects/quantitative-research-stock-trend/models/aapl_tcn.pt
Device: cpu
Model loaded: PASSED


In [40]:
WINDOW_SIZE = 22

TEST_START_DATE = pd.Timestamp(
    "2026-06-12"
)

print("=" * 60)
print("BACKTEST MODEL CONFIGURATION")
print("=" * 60)

print("Input window:", WINDOW_SIZE)
print("Test start:", TEST_START_DATE.date())
print("Feature count:", len(FEATURE_COLUMNS))

assert WINDOW_SIZE == 22
assert len(FEATURE_COLUMNS) == 9

print("Configuration validation: PASSED")

BACKTEST MODEL CONFIGURATION
Input window: 22
Test start: 2026-06-12
Feature count: 9
Configuration validation: PASSED


In [41]:
print(os.listdir(MODEL_DIR))

['aapl_tcn.pt', 'aapl_scaler.joblib']


In [42]:
import joblib

SCALER_PATH = os.path.join(
    MODEL_DIR,
    "aapl_scaler.joblib"
)

assert os.path.exists(
    SCALER_PATH
), f"Missing scaler: {SCALER_PATH}"

scaler = joblib.load(
    SCALER_PATH
)

print("=" * 60)
print("TRAINING SCALER LOADED")
print("=" * 60)

print("Scaler path:", SCALER_PATH)
print("Scaler type:", type(scaler).__name__)

assert hasattr(scaler, "transform")

print("Scaler loading: PASSED")

TRAINING SCALER LOADED
Scaler path: /Users/senuka/Documents/Projects/My Projects/quantitative-research-stock-trend/models/aapl_scaler.joblib
Scaler type: StandardScaler
Scaler loading: PASSED


In [43]:
assert scaler.n_features_in_ == 9

print("=" * 60)
print("SCALER VALIDATION")
print("=" * 60)

print(
    "Features fitted:",
    scaler.n_features_in_
)

print("Training-only scaler: PASSED")
print("Feature dimension: PASSED")

SCALER VALIDATION
Features fitted: 9
Training-only scaler: PASSED
Feature dimension: PASSED


In [44]:
model_features = aapl_model[
    FEATURE_COLUMNS
].copy()

model_dates = aapl_model[
    "Date"
].copy()

print("=" * 60)
print("MODEL FEATURE MATRIX")
print("=" * 60)

print("Shape:", model_features.shape)

print(
    "Date range:",
    model_dates.min().date(),
    "→",
    model_dates.max().date()
)

print("Feature count:", model_features.shape[1])

assert model_features.shape[1] == 9
assert model_features.isna().sum().sum() == 0

print("Feature matrix validation: PASSED")

MODEL FEATURE MATRIX
Shape: (232, 9)
Date range: 2025-09-15 → 2026-08-17
Feature count: 9
Feature matrix validation: PASSED


In [45]:
scaled_features = scaler.transform(
    model_features
)

scaled_features = np.asarray(
    scaled_features,
    dtype=np.float32
)

print("=" * 60)
print("FEATURE SCALING")
print("=" * 60)

print("Scaled shape:", scaled_features.shape)

print(
    "All values finite:",
    np.isfinite(scaled_features).all()
)

assert scaled_features.shape == (
    len(model_features),
    9
)

assert np.isfinite(
    scaled_features
).all()

print("Scaling validation: PASSED")

FEATURE SCALING
Scaled shape: (232, 9)
All values finite: True
Scaling validation: PASSED


In [46]:
test_mask = (
    model_dates >= TEST_START_DATE
)

test_indices = np.where(
    test_mask
)[0]

assert len(test_indices) > 0

X_test_windows = []
prediction_dates = []

for index in test_indices:

    if index < WINDOW_SIZE - 1:
        continue

    window_start = (
        index - WINDOW_SIZE + 1
    )

    window = scaled_features[
        window_start:index + 1
    ]

    if window.shape != (
        WINDOW_SIZE,
        9
    ):
        continue

    X_test_windows.append(
        window
    )

    prediction_dates.append(
        model_dates.iloc[index]
    )

X_test_windows = np.asarray(
    X_test_windows,
    dtype=np.float32
)

prediction_dates = pd.to_datetime(
    prediction_dates
)

print("=" * 60)
print("TEST WINDOWS")
print("=" * 60)

print(
    "X test shape:",
    X_test_windows.shape
)

print(
    "Prediction dates:",
    len(prediction_dates)
)

print(
    "First prediction:",
    prediction_dates.min().date()
)

print(
    "Last prediction:",
    prediction_dates.max().date()
)

assert X_test_windows.ndim == 3

assert X_test_windows.shape[1] == 22

assert X_test_windows.shape[2] == 9

assert len(X_test_windows) == len(
    prediction_dates
)

print("Window construction: PASSED")

TEST WINDOWS
X test shape: (45, 22, 9)
Prediction dates: 45
First prediction: 2026-06-12
Last prediction: 2026-08-17
Window construction: PASSED


In [47]:
first_prediction_date = prediction_dates[0]

first_prediction_index = (
    model_dates[
        model_dates == first_prediction_date
    ].index[0]
)

window_start_index = (
    first_prediction_index
    - WINDOW_SIZE
    + 1
)

window_dates = model_dates.iloc[
    window_start_index:
    first_prediction_index + 1
]

print("=" * 60)
print("FIRST TEST WINDOW VALIDATION")
print("=" * 60)

print(
    "Context start:",
    window_dates.iloc[0].date()
)

print(
    "Prediction date:",
    window_dates.iloc[-1].date()
)

print(
    "Window length:",
    len(window_dates)
)

assert len(window_dates) == 22

assert window_dates.is_monotonic_increasing

assert window_dates.iloc[-1] == first_prediction_date

print("22-day temporal window: PASSED")
print("Chronological order: PASSED")

FIRST TEST WINDOW VALIDATION
Context start: 2026-05-13
Prediction date: 2026-06-12
Window length: 22
22-day temporal window: PASSED
Chronological order: PASSED


In [48]:
X_test_tensor = torch.tensor(
    X_test_windows,
    dtype=torch.float32,
    device=device
)

print("=" * 60)
print("TEST TENSOR")
print("=" * 60)

print(
    "Tensor shape:",
    X_test_tensor.shape
)

print(
    "Device:",
    X_test_tensor.device
)

print(
    "Dtype:",
    X_test_tensor.dtype
)

assert X_test_tensor.shape[1:] == (
    22,
    9
)

assert torch.isfinite(
    X_test_tensor
).all()

print("Tensor validation: PASSED")

TEST TENSOR
Tensor shape: torch.Size([45, 22, 9])
Device: cpu
Dtype: torch.float32
Tensor validation: PASSED
